# r.dem worked example

This notebook builds the synthetic scene used by the *r.dem* manual pages,
runs the whole toolset on it, and writes every figure the manual references.
Re-running it top to bottom regenerates all of them.

The scene is built on `elev_lid792_1m` from the GRASS North Carolina sample
dataset, so the example is reproducible without any private data. A known
change surface is added with
[r.earthworks](https://grass.osgeo.org/grass-devel/manuals/addons/r.earthworks.html),
a known rigid offset is applied, and known systematic bias fields are added.
Because every quantity is known in advance, each tool's output can be checked
against the answer.

## Requirements

- GRASS 8.5 or newer, started in the `nc_spm_full_v2alpha2` project
- the *r.dem* toolset and the *r.earthworks* addon
- `matplotlib` and `pillow` for the figure panels

```sh
g.extension extension=r.earthworks
g.extension extension=r.dem
```

In [1]:
import os
import pathlib
import subprocess
import tempfile
from io import StringIO

import grass.jupyter as gj
import matplotlib.pyplot as plt
from grass.tools import Tools
from PIL import Image

# One Tools object drives every GRASS call in this notebook. capture_stderr
# collects the reports the tools print, consistent_return_value makes even
# the tools that print nothing return a result object, and overwrite lets
# the notebook be re-run from the top.
tools = Tools(overwrite=True, capture_stderr=True, consistent_return_value=True)

# Scratch space for the CSV and transform files the tools write.
TMPDIR = pathlib.Path(tempfile.mkdtemp(prefix="r_dem_examples_"))

# Where the manual figures are written: one per tool directory. The notebook
# lives in the toolbox directory, so this resolves without configuration.
TOOLBOX = os.environ.get("R_DEM_TOOLBOX", os.getcwd())

# GRASS style guide: manual images are 600 px wide.
FIGURE_WIDTH = 600

## Helpers

`render` draws one GRASS map to a PNG panel. `panel_figure` stacks panels
into a single figure sized to the 600 px the GRASS style guide asks for.

In [2]:
def render(name, layers, width=600, height=640, legend=None):
    """Render layers to a PNG and return its path.

    Each layer is a (module, kwargs) pair passed straight to the display
    module, for example ("d.rast", {"map": "elevation"}).
    """
    path = os.path.join(str(TMPDIR), f"{name}.png")
    plot = gj.Map(width=width, height=height, filename=path, use_region=True)
    for module, kwargs in layers:
        plot.run(module, **kwargs)
    if legend:
        plot.run("d.legend", **legend)
    return path


# Named color schemes. Each is a list of (position, "r:g:b") stops, with
# the position given as a fraction of whatever range the scheme is stretched
# over. Holding them in one table means the GRASS ramp and the matplotlib
# colorbar are built from the same numbers and cannot drift apart.
COLOR_SCHEMES = {
    # Erosion red, deposition blue, after the table used for GRASS erosion
    # modeling: erosion runs yellow to orange to red to magenta as it
    # deepens, deposition runs cyan to teal to blue, and near-zero is pale
    # green. The published breaks are in units of erosion/deposition * 10,
    # so they are held here as fractions and rescaled to metres on use.
    "erosion_deposition": [
        (0.00, "#FF00FF"),
        (0.20, "#FF0000"),
        (0.35, "#FF7F00"),
        (0.45, "#FFFF00"),
        (0.50, "#C8FFC8"),
        (0.55, "#00FFFF"),
        (0.65, "#00BFBF"),
        (0.80, "#0000FF"),
        (1.00, "#000080"),
    ],
    # Pale where change is easy to detect, dark where it is not.
    "detection_limit": [
        (0.00, "#FFF5EB"),
        (0.50, "#E6823C"),
        (1.00, "#78140A"),
    ],
    # Perceptually uniform, for z-scores and other one-sided quantities.
    "sequential": [
        (0.00, "#440154"),
        (0.50, "#21918C"),
        (1.00, "#FDE725"),
    ],
    # Terrain, stretched across a pair of surfaces so both share one ramp.
    "elevation": [
        (0.00, "#005000"),
        (0.25, "#3CA03C"),
        (0.50, "#E6DC78"),
        (0.75, "#B4783C"),
        (1.00, "#E1CDB9"),
    ],
}

NULL_COLOR = "#F0F0F0"


def raster_info(raster):
    """Range and metadata of a raster, as typed values."""
    return tools.r_info(map=raster, flags="gr").keyval


def univar(raster, **kwargs):
    """Univariate statistics of a raster, including the extended ones."""
    return tools.r_univar(map=raster, flags="e", format="json", **kwargs).json


def color_scheme(scheme_name, low, high, clamp_to=None):
    """Color rules for a named scheme stretched over a value range.

    Returns an io.StringIO, which grass.tools converts to `rules=-` and
    passes on standard input.

    clamp_to extends the outermost stops to that raster's range, so values
    beyond the scheme saturate rather than rendering white, which would read
    as no data.
    """
    stops = [
        (low + fraction * (high - low), color)
        for fraction, color in COLOR_SCHEMES[scheme_name]
    ]
    if clamp_to:
        info = raster_info(clamp_to)
        if info["min"] < stops[0][0]:
            stops.insert(0, (info["min"], stops[0][1]))
        if info["max"] > stops[-1][0]:
            stops.append((info["max"], stops[-1][1]))
    rules = "\n".join(f"{value:g} {color}" for value, color in stops)
    return StringIO(f"{rules}\nnv {NULL_COLOR}\n")


def apply_scheme(maps, scheme_name, low, high, clamp=True):
    """Apply a named color scheme to one or more maps."""
    clamp_to = maps if clamp and "," not in maps else None
    tools.r_colors(map=maps, rules=color_scheme(scheme_name, low, high, clamp_to))


def scheme_cmap(scheme_name):
    """Matplotlib colormap for a named scheme, for the shared colorbar."""
    from matplotlib.colors import LinearSegmentedColormap

    return LinearSegmentedColormap.from_list(
        scheme_name,
        COLOR_SCHEMES[scheme_name],
    )


def panel_figure(
    panels, out_path, titles=None, ncols=None, dpi=100, colorbar=None, cbar_label=""
):
    """Combine rendered panels into one figure, written at FIGURE_WIDTH px.

    colorbar is (vmin, vmax, cmap) and adds one shared scale bar below the
    panels, which is what makes the panels readable as a comparison.
    """
    ncols = ncols or len(panels)
    nrows = (len(panels) + ncols - 1) // ncols
    images = [Image.open(p) for p in panels]
    aspect = images[0].height / images[0].width
    fig_w = FIGURE_WIDTH / dpi
    fig_h = fig_w * aspect * nrows / ncols
    if colorbar:
        fig_h += 0.45
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), dpi=dpi)
    axes = [axes] if len(panels) == 1 else list(axes.flat)
    for ax, image, title in zip(axes, images, titles or [None] * len(panels)):
        ax.imshow(image)
        ax.set_axis_off()
        if title:
            ax.set_title(title, fontsize=7)
    for ax in axes[len(panels) :]:
        ax.set_axis_off()
    fig.tight_layout(pad=0.3)
    if colorbar:
        vmin, vmax, cmap = colorbar
        fig.subplots_adjust(bottom=0.20)
        cax = fig.add_axes([0.25, 0.09, 0.5, 0.045])
        bar = fig.colorbar(
            plt.cm.ScalarMappable(norm=plt.Normalize(vmin=vmin, vmax=vmax), cmap=cmap),
            cax=cax,
            orientation="horizontal",
        )
        bar.set_label(cbar_label, fontsize=7)
        bar.ax.tick_params(labelsize=6)
    fig.savefig(out_path, dpi=dpi, bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)
    # Enforce the 600 px width and strip metadata.
    subprocess.run(
        [
            "mogrify",
            "-resize",
            f"{FIGURE_WIDTH}x",
            "-strip",
            "-define",
            "png:compression-level=9",
            out_path,
        ],
        check=True,
    )
    return out_path

## 1. The scene

### 1.1 Pre-event reference and the known change

`elev_lid792_1m` is the pre-event reference. *r.earthworks* cuts a scour pit
and builds a deposition fan on it, and reports the volume it moved. Those
reported volumes are the truth that *r.dem.change* has to recover at the end.

In [3]:
tools.g_region(raster="elev_lid792_1m")

# Derive the drainage network: the change follows it, as a flood's would.
tools.r_watershed(
    elevation="elev_lid792_1m",
    accumulation="accumulation",
    overwrite=True,
)
tools.r_stream_extract(
    elevation="elev_lid792_1m",
    accumulation="accumulation",
    threshold=8000,
    stream_raster="streams",
    overwrite=True,
)

# Scour the upper reach, deposit on the lower one. Both are kept out of
# forest on purpose: r.dem.bias method=forest cannot tell a canopy bump from
# real deposition, so change under canopy would be erased by that stage.
for name, test in (
    ("channel_scour", "elev_lid792_1m > 116"),
    ("channel_fill", "elev_lid792_1m < 110"),
):
    tools.r_mapcalc(
        expression=f"{name}_r = if(streams && {test} && landcover_1m != 2, 1, null())",
        overwrite=True,
    )
    tools.r_to_vect(
        flags="s",
        input=f"{name}_r",
        output=name,
        type="line",
        overwrite=True,
    )

cut_report = tools.r_earthworks(
    flags="p",
    elevation="elev_lid792_1m",
    earthworks="dsm_scoured",
    operation="cut",
    mode="relative",
    lines="channel_scour",
    z=-2.5,
    function="linear",
    linear=0.35,
    flat=5,
    nprocs=1,
    overwrite=True,
)
fill_report = tools.r_earthworks(
    flags="p",
    elevation="dsm_scoured",
    earthworks="dsm_event",
    operation="fill",
    mode="relative",
    lines="channel_fill",
    z=2.0,
    function="linear",
    linear=0.30,
    flat=10,
    nprocs=1,
    overwrite=True,
)
cut = cut_report.stderr
fill = fill_report.stderr
print(cut)
print(fill)

tools.r_mapcalc(expression="change_truth = dsm_event - elev_lid792_1m", overwrite=True)

TRUTH = {}
for line in (cut + fill).splitlines():
    if line.startswith("Net cut:"):
        TRUTH["erosion"] = abs(float(line.split(":")[1].split()[0]))
    elif line.startswith("Net fill:"):
        TRUTH["deposition"] = float(line.split(":")[1].split()[0])
TRUTH["net"] = TRUTH["deposition"] - TRUTH["erosion"]
TRUTH

   0%   3%   6%   9%  12%  15%  18%  21%  24%  27%  30%  33%  36%  39%  42%  45%  48%  51%  54%  57%  60%  63%  66%  69%  72%  75%  78%  81%  84%  87%  90%  93%  96%  99% 100%
Using quadtree segmentation for faster computation. If artifacts occur,
increase the size of the border.
   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   0%   3%   0%   3%   0%   4%   0%   5%   0%   0%   5%   8%  11%  14%  17%  20%  23%  26%  29%  32%  35%  38%  41%  44%  47%  50%  53%  56%  59%  62

{'erosion': 32667.1072769316,
 'deposition': 16323.9426829657,
 'net': -16343.164593965901}

### 1.2 The survey error model

A photogrammetric surface differs from the lidar reference by more than the
change of interest. Three error components are added here, each of which one
*r.dem.bias* method is designed to remove:

- **doming**, a smooth long-wavelength surface, removed by `method=spline`
- a **roughness-correlated** term, removed by `method=regression`
- a **canopy bump** over forest, removed by `method=forest`

plus uncorrelated noise, which sets the level of detection.

In [4]:
tools.r_dem_stats(
    input="elev_lid792_1m",
    output="roughness",
    metric="roughness_std",
    window=13,
    overwrite=True,
)

tools.r_mapcalc(
    expression="bias_dome = 0.25 * (1 - ((x() - 638650) * (x() - 638650)"
    " + (y() - 220375) * (y() - 220375)) / (400.0 * 400.0))",
    overwrite=True,
)
tools.r_mapcalc(expression="bias_rough = 0.6 * roughness", overwrite=True)
tools.r_mapcalc(
    expression="bias_canopy = if(landcover_1m == 2, 1.5, 0)", overwrite=True
)
tools.r_mapcalc(
    expression="bias_truth = bias_dome + bias_rough + bias_canopy", overwrite=True
)
# Survey noise is heteroscedastic: it grows with surface roughness and again
# under canopy. A single detection limit cannot serve both the smooth fields
# and the forest, which is what makes method=local worth having.
tools.r_surf_gauss(
    output="noise_unit",
    mean=0,
    sigma=1,
    seed=42,
    overwrite=True,
)
tools.r_mapcalc(
    expression="sigma_survey = 0.05 + 0.20 * roughness"
    " + if(landcover_1m == 2, 0.08, 0)",
    overwrite=True,
)
tools.r_mapcalc(expression="noise = noise_unit * sigma_survey", overwrite=True)

ToolResult(returncode=0, stdout='', stderr='')

### 1.3 Two post-event surfaces

`dsm_offset` carries a rigid misregistration and nothing else that would
confuse it: a horizontal shift of 0.65 m (0.4596 m on each axis) and a
vertical shift of 1.32 m. It is what the co-registration tools are run on,
and the applied offset is the answer they have to return.

`dsm_post` is already aligned and carries the survey error model. It is what
the bias, level-of-detection, and change tools are run on.

The two are kept separate because a long-wavelength vertical bias is partly
degenerate with a horizontal shift in the first-order Nuth and Kaab model, so
a surface carrying both has no single correct co-registration answer.

In [5]:
DX = DY = 0.4596
DZ = 1.32

region = tools.g_region(flags="g").keyval
tools.g_copy(raster=("dsm_event", "tmp_shift"), overwrite=True)
tools.r_region(
    map="tmp_shift",
    n=region["n"] + DY,
    s=region["s"] + DY,
    e=region["e"] + DX,
    w=region["w"] + DX,
)
tools.g_region(raster="elev_lid792_1m")
tools.r_resamp_interp(
    input="tmp_shift",
    output="post_shift",
    method="bilinear",
    overwrite=True,
)

tools.r_mapcalc(
    expression=f"dsm_offset = post_shift + {DZ} + bias_canopy + noise", overwrite=True
)
tools.r_mapcalc(expression="dsm_post = dsm_event + bias_truth + noise", overwrite=True)
tools.r_mapcalc(expression="dod_raw = dsm_post - elev_lid792_1m", overwrite=True)

ToolResult(returncode=0, stdout='', stderr='')

### 1.4 Masks

`stable_terrain` is the broad, sloped, unchanged terrain the Nuth and Kaab
step and the bias fits need. `pgcp_roads` is the flat road network the PGCP
vertical step needs. They are different masks for different jobs, as
*r.dem.coregister* explains.

In [6]:
tools.r_mapcalc(
    expression="change_foot = if(abs(change_truth) > 0.05, 1, null())", overwrite=True
)
tools.r_mapcalc(
    expression="stable_terrain = if(landcover_1m != 2 && landcover_1m != 1"
    " && isnull(change_foot), 1, null())",
    overwrite=True,
)
tools.r_mapcalc(expression="forest = if(landcover_1m == 2, 1, null())", overwrite=True)
# Forest is unchanged terrain, so it is stable for estimating uncertainty
# even though the canopy bump makes it unusable for the Nuth and Kaab
# regression. Leaving it out here would leave the noisiest part of the map
# uncharacterised, and any fallback would then under-protect it.
tools.r_mapcalc(
    expression="stable_lod = if(landcover_1m != 1 && isnull(change_foot), 1, null())",
    overwrite=True,
)
tools.v_clip(
    flags="r",
    input="streets_wake",
    output="pgcp_roads",
    overwrite=True,
)

ToolResult(returncode=0, stdout='', stderr='
--------------------------------------------------------------------------------
Clipping by region.
--------------------------------------------------------------------------------
Building topology for vector map <temp_1640239@rdem_nb>...
Registering primitives...

Building areas...
   0% 100%
Attaching islands...
   0% 100%
Attaching centroids...
   0% 100%
Copying vector features from <streets_wake@PERMANENT>...
   1%   3%   5%   7%   9%  11%  13%  15%  17%  19%  21%  23%  25%  27%  29%  31%  33%  35%  37%  39%  41%  43%  45%  47%  49%  51%  53%  55%  57%  59%  61%  63%  65%  67%  69%  71%  73%  75%  77%  79%  81%  83%  85%  87%  89%  91%  93%  95%  97%  99% 100%

In [7]:
# The two surfaces must share one elevation ramp, or a 3 m change on a 28 m
# range looks like a different landscape.
# One elevation ramp spanning both surfaces. Copying the reference table
# would leave the canopy-bumped cells of dsm_post outside its range, and
# out-of-range cells render white.
pre = raster_info("elev_lid792_1m")
post = raster_info("dsm_post")
elev_low = min(pre["min"], post["min"])
elev_high = max(pre["max"], post["max"])
apply_scheme("elev_lid792_1m,dsm_post", "elevation", elev_low, elev_high, clamp=False)
print(f"shared elevation ramp: {elev_low:.1f} to {elev_high:.1f} m")

apply_scheme("change_truth", "erosion_deposition", -2.5, 2.5)

panel_figure(
    [
        render("pre", [("d.rast", {"map": "elev_lid792_1m"})]),
        render("post", [("d.rast", {"map": "dsm_post"})]),
        render("truth", [("d.rast", {"map": "change_truth"})]),
    ],
    os.path.join(TOOLBOX, "r_dem_scene.png"),
    titles=[
        "Pre-event reference, 103.8 to 131.6 m",
        "Post-event surface, same ramp",
        "Known change, -2.5 to +2.0 m",
    ],
)

shared elevation ramp: 103.8 to 133.4 m


'/home/coreywhite/Documents/GitHub/cwhite911/r-dem/src/raster/r.dem/r_dem_scene.png'

## 2. Co-registration

Run on `dsm_offset`, whose offset is known: `dx = dy = 0.4596`, `dz = 1.32`.

In [8]:
nk_transform = os.path.join(str(TMPDIR), "nk.txt")
print(
    tools.r_dem_nk(
        sfm="dsm_offset",
        lidar="elev_lid792_1m",
        stable_mask="stable_terrain",
        output="nk_out",
        transform_output=nk_transform,
        overwrite=True,
    ).stderr
)
print(open(nk_transform).read())

Reading input rasters into memory...
Computing slope, aspect, and predictors...
Outer pass 0: cumulative dz=1.3195 dx=0.4609 dy=0.4574 (increment
|dxy|=0.6494)
Outer pass 1: cumulative dz=1.3195 dx=0.4621 dy=0.4564 (increment
|dxy|=0.0016)
Converged transform: dz=1.319499 dx=0.462080 dy=0.456402
Applying vertical correction (dz) and horizontal translation (dx, dy)...
Done. Output raster map <nk_out>.
Residual raster map <nk_out_resid>.

# r.dem.nk transform
dz=1.3194993702
dx=0.4620796677
dy=0.4564018614



In [9]:
icp_transform = os.path.join(str(TMPDIR), "icp.txt")
tools.r_dem_icp(
    source="dsm_offset",
    reference="elev_lid792_1m",
    mask="stable_terrain",
    output="icp_out",
    dof=4,
    transform_out=icp_transform,
    overwrite=True,
)
print(open(icp_transform).read())

# r.dem.icp transform (dof=4)
tx=-0.4830902607
ty=-0.4542042745
tz=-1.3189444147
yaw=0.0000056765
roll=0.0000000000
pitch=0.0000000000



In [10]:
print(
    tools.r_dem_coregister(
        dem="dsm_offset",
        reference="elev_lid792_1m",
        pgcp="pgcp_roads",
        stable_mask="stable_terrain",
        output="coreg_out",
        method="nk",
        overwrite=True,
    ).stderr
)

   0%   3%   6%   9%  12%  15%  18%  21%  24%  27%  30%  33%  36%  39%  42%  45%  48%  51%  54%  57%  60%  63%  66%  69%  72%  75%  78%  81%  84%  87%  90%  93%  96%  99% 100%
PGCP vertical correction:
N samples: 9725
Median bias: 1.3124 m
NMAD: 0.0798 m
RMSE: 1.3065 m
Reading input rasters into memory...
Computing slope, aspect, and predictors...
Outer pass 0: cumulative dz=0.0071 dx=0.4609 dy=0.4574 (increment
|dxy|=0.6494)
Outer pass 1: cumulative dz=0.0071 dx=0.4621 dy=0.4564 (increment
|dxy|=0.0016)
Converged transform: dz=0.007051 dx=0.462080 dy=0.456402
Applying vertical correction (dz) and horizontal translation (dx, dy)...
Done. Output raster map <coreg_out>.
Residual raster map <coreg_out_resid>.



In [11]:
# Show the whole residual, not just the masked part, and hold the color
# range fixed across the three panels so they can be compared.
for name, source in (
    ("resid_before", "dsm_offset"),
    ("resid_nk", "nk_out"),
    ("resid_icp", "icp_out"),
):
    tools.r_mapcalc(expression=f"{name} = {source} - elev_lid792_1m", overwrite=True)
    apply_scheme(name, "erosion_deposition", -2, 2)

panel_figure(
    [
        render("rb", [("d.rast", {"map": "resid_before"})]),
        render("rn", [("d.rast", {"map": "resid_nk"})]),
        render("ri", [("d.rast", {"map": "resid_icp"})]),
    ],
    os.path.join(TOOLBOX, "r.dem.coregister", "r_dem_coregister_residuals.png"),
    titles=[
        "Before: 0.65 m horizontal, 1.32 m vertical",
        "After r.dem.nk",
        "After r.dem.icp (dof=4)",
    ],
    colorbar=(-2, 2, scheme_cmap("erosion_deposition")),
    cbar_label="elevation residual (m): erosion red, deposition blue",
)

'/home/coreywhite/Documents/GitHub/cwhite911/r-dem/src/raster/r.dem/r.dem.coregister/r_dem_coregister_residuals.png'

In [12]:
# The per-iteration convergence r.dem.nk reports.
lines = [
    line
    for line in tools.r_dem_nk(
        sfm="dsm_offset",
        lidar="elev_lid792_1m",
        stable_mask="stable_terrain",
        output="nk_out",
        overwrite=True,
    ).stderr.splitlines()
    if "Outer pass" in line
]
print("\n".join(lines))

Outer pass 0: cumulative dz=1.3195 dx=0.4609 dy=0.4574 (increment
Outer pass 1: cumulative dz=1.3195 dx=0.4621 dy=0.4564 (increment


In [13]:
panel_figure(
    [
        render("nk_before", [("d.rast", {"map": "resid_before"})]),
        render("nk_after", [("d.rast", {"map": "resid_nk"})]),
    ],
    os.path.join(TOOLBOX, "r.dem.nk", "r_dem_nk_convergence.png"),
    titles=["Residual before", "After r.dem.nk"],
    colorbar=(-2, 2, scheme_cmap("erosion_deposition")),
    cbar_label="elevation residual (m): erosion red, deposition blue",
)
panel_figure(
    [
        render("icp_before", [("d.rast", {"map": "resid_before"})]),
        render("icp_after", [("d.rast", {"map": "resid_icp"})]),
    ],
    os.path.join(TOOLBOX, "r.dem.icp", "r_dem_icp_alignment.png"),
    titles=["Residual before", "After r.dem.icp (dof=4)"],
    colorbar=(-2, 2, scheme_cmap("erosion_deposition")),
    cbar_label="elevation residual (m): erosion red, deposition blue",
)

'/home/coreywhite/Documents/GitHub/cwhite911/r-dem/src/raster/r.dem/r.dem.icp/r_dem_icp_alignment.png'

## 3. Bias removal

Run on `dod_raw`. The spline stage takes out the doming, the forest stage
takes out the canopy bump. Chain them: neither alone removes both.

In [14]:
tools.r_dem_bias(
    dod="dod_raw",
    output="dod_spline",
    method="spline",
    stable_mask="stable_terrain",
    bias_field="bias_spline",
    overwrite=True,
)
tools.r_dem_bias(
    dod="dod_spline",
    output="dod_debiased",
    method="forest",
    mask="forest",
    window=21,
    overwrite=True,
)

for name, mask in (
    ("dod_raw", "stable_terrain"),
    ("dod_debiased", "stable_terrain"),
    ("dod_raw", "forest"),
    ("dod_debiased", "forest"),
):
    tools.r_mapcalc(expression=f"tmp_chk = if({mask}, {name}, null())", overwrite=True)
    stats = univar("tmp_chk")
    print(f"{name:14s} on {mask:15s} median = {stats['median']: .4f} m")

dod_raw        on stable_terrain  median =  0.2105 m


dod_debiased   on stable_terrain  median =  0.0002 m


dod_raw        on forest          median =  1.6805 m


dod_debiased   on forest          median =  0.0003 m


In [15]:
for name in ("dod_raw", "bias_spline", "dod_debiased"):
    apply_scheme(name, "erosion_deposition", -2, 2)
panel_figure(
    [
        render("b_raw", [("d.rast", {"map": "dod_raw"})]),
        render("b_field", [("d.rast", {"map": "bias_spline"})]),
        render("b_deb", [("d.rast", {"map": "dod_debiased"})]),
    ],
    os.path.join(TOOLBOX, "r.dem.bias", "r_dem_bias_methods.png"),
    titles=[
        "Raw DoD: survey bias plus real change",
        "Fitted spline bias field",
        "Debiased DoD: only the real change",
    ],
    colorbar=(-2, 2, scheme_cmap("erosion_deposition")),
    cbar_label="elevation difference (m): erosion red, deposition blue",
)

'/home/coreywhite/Documents/GitHub/cwhite911/r-dem/src/raster/r.dem/r.dem.bias/r_dem_bias_methods.png'

## 4. Terrain metrics

In [16]:
for metric, window in (
    ("slope", None),
    ("roughness_std", 13),
    ("diversity_geomorphon", 13),
):
    kwargs = {"window": window} if window else {}
    tools.r_dem_stats(
        input="elev_lid792_1m",
        output=f"st_{metric}",
        metric=metric,
        overwrite=True,
        **kwargs,
    )


# The three metrics carry different units, so each panel needs its own
# legend rather than one shared bar.
def with_legend(name):
    return [
        ("d.rast", {"map": name}),
        ("d.legend", {"raster": name, "at": "55,95,3,8", "flags": "b", "fontsize": 13}),
    ]


panel_figure(
    [
        render("s1", with_legend("st_slope")),
        render("s2", with_legend("st_roughness_std")),
        render("s3", with_legend("st_diversity_geomorphon")),
    ],
    os.path.join(TOOLBOX, "r.dem.stats", "r_dem_stats_metrics.png"),
    titles=[
        "slope (degrees)",
        "roughness_std (m, window=13)",
        "diversity_geomorphon (window=13)",
    ],
)

'/home/coreywhite/Documents/GitHub/cwhite911/r-dem/src/raster/r.dem/r.dem.stats/r_dem_stats_metrics.png'

## 5. Level of detection

The debiased residual on stable terrain is the injected noise, so the global
LoD comes out at `z(0.95)` times its NMAD.

In [17]:
print(
    tools.r_dem_lod(
        dod="dod_debiased",
        output="lod_global",
        method="global",
        stable_mask="stable_lod",
        confidence=0.95,
        overwrite=True,
    ).stderr
)
print(
    tools.r_dem_lod(
        dod="dod_debiased",
        output="lod_local",
        method="local",
        window=21,
        stable_mask="stable_lod",
        output_sigma="sigma_combined",
        output_domain="lod_domain",
        confidence=0.95,
        overwrite=True,
    ).stderr
)

# No stable cell falls inside the window in the interior of the change
# features, so the local limit is undefined exactly where the change is.
# Fall back to the flight-wide limit there.
tools.r_mapcalc(
    expression="lod_filled = if(isnull(lod_local), lod_global, lod_local)",
    overwrite=True,
)

Estimating NMAD from stable pixels...
Estimated NMAD: 0.0882 m
Global LoD (95% CI):
NMAD: 0.0882 m
sigma: 0.0882 m
z: 1.9600
LoD: 0.1729 m (uniform)



Significance domain: 502978 of 510600 observed cells (98.5%)
Local LoD stats:
Min: 0.0585 m
Mean: 0.1749 m
Max: 0.4584 m
CV: 30.6%



ToolResult(returncode=0, stdout='', stderr='')

In [18]:
# Put both methods on one scale. The global panel is deliberately flat:
# that is the point of the comparison. White is where no stable cell falls
# inside the window, so the local LoD is undefined there.
# Clamp the ramp to the central 90% of the values. Stretching it over the
# full min-max puts almost every cell in one shade.
spread = univar("lod_local", percentile=[5, 95])
percentiles = {p["percentile"]: p["value"] for p in spread["percentiles"]}
low = round(percentiles[5], 2)
high = round(percentiles[95], 2)
apply_scheme("lod_local", "detection_limit", low, high)
apply_scheme("lod_global", "detection_limit", low, high)
uniform = raster_info("lod_global")["min"]
print(
    f"local LoD: {low} to {high} m over the central 90%, uniform global {uniform:.3f} m"
)

panel_figure(
    [
        render("l1", [("d.rast", {"map": "lod_local"})]),
        render("l2", [("d.rast", {"map": "lod_global"})]),
    ],
    os.path.join(TOOLBOX, "r.dem.lod", "r_dem_lod_local.png"),
    titles=[
        f"method=local, window=21, {low} to {high} m",
        f"method=global, uniform {uniform:.3f} m",
    ],
    colorbar=(low, high, scheme_cmap("detection_limit")),
    cbar_label="Level of Detection (m), clamped to the central 90%",
)

local LoD: 0.12 to 0.31 m over the central 90%, uniform global 0.173 m


'/home/coreywhite/Documents/GitHub/cwhite911/r-dem/src/raster/r.dem/r.dem.lod/r_dem_lod_local.png'

## 6. Change detection

The volumes *r.dem.change* reports are compared with the volumes
*r.earthworks* moved. They differ by a few percent because the LoD mask
discards the low tails of both features.

In [19]:
volume_csv = os.path.join(str(TMPDIR), "volumes.csv")
print(
    tools.r_dem_change(
        dod="dod_debiased",
        lod="lod_filled",
        output_sig="dod_significant",
        volume_csv=volume_csv,
        flags="n",
        overwrite=True,
    ).stderr
)

recovered = {}
with open(volume_csv) as handle:
    next(handle)
    for line in handle:
        fields = line.strip().split(",")
        recovered[fields[0]] = float(fields[1])

print(f"{'':12s}{'truth':>14s}{'recovered':>14s}{'ratio':>9s}")
for key in ("erosion", "deposition"):
    print(
        f"{key:12s}{TRUTH[key]:14.1f}{recovered[key]:14.1f}"
        f"{recovered[key] / TRUTH[key]:9.3f}"
    )

Removed isolated significant cells (speckle)
   2%   5%   8%  11%  14%  17%  20%  23%  26%  29%  35%  32%  38%  41%  44%  47%  50%  53%  56%  59%  62%  65%  68%  71%  74%  77%  80%  83%  86%  89%  92%  95%  98% 100%
 100%
   2%   5%   8%  11%  14%  17%  20%  23%  26%  29%  32%  35%  38%  41%  41%  44%  47%  50%  53%  56%  59%  62%  65%  68%  71%  74%  77%  80%  83%  86%  89%  92%  95%  98% 100%
 100%
Volumetric summary (significant cells only):
Deposition: 17,149.0 m3 (16,455 cells)
Erosion: 33,271.1 m3 (23,816 cells)
Net: -16,122.1 m3
Net (yd3): -21,087.7
Volume stats:
/tmp/grass8-coreywhite-1635553/r_dem_examples__lbpcn0p/volumes.csv



In [20]:
apply_scheme("dod_significant", "erosion_deposition", -2.5, 2.5)
apply_scheme("change_truth", "erosion_deposition", -2.5, 2.5)
panel_figure(
    [
        render("c1", [("d.rast", {"map": "change_truth"})]),
        render("c2", [("d.rast", {"map": "dod_significant"})]),
    ],
    os.path.join(TOOLBOX, "r.dem.change", "r_dem_change_volumes.png"),
    titles=["Known change (r.earthworks)", "Significant DoD above the LoD"],
    colorbar=(-2.5, 2.5, scheme_cmap("erosion_deposition")),
    cbar_label="elevation change (m): erosion red, deposition blue",
)

'/home/coreywhite/Documents/GitHub/cwhite911/r-dem/src/raster/r.dem/r.dem.change/r_dem_change_volumes.png'

## 7. Error propagation

In [21]:
# The local sigma is undefined wherever no stable cell falls inside the
# window, which includes the interior of the change features themselves.
# Those cells are exactly the ones worth testing, so fall back to the
# flight-wide sigma there rather than leaving them untestable.
global_sigma = raster_info("lod_global")["min"] / 1.96
tools.r_mapcalc(
    expression=f"sigma_filled = if(isnull(sigma_combined), {global_sigma},"
    " sigma_combined)",
    overwrite=True,
)
print(f"global sigma used as fallback: {global_sigma:.4f} m")

print(
    tools.r_dem_errprop(
        dod="dod_debiased",
        sigma="sigma_filled",
        output_sigma="sigma_dod",
        output_lod="lod_errprop",
        output_zscore="zscore",
        output_class="significance",
        confidence=0.95,
        overwrite=True,
    ).stderr
)
print(tools.r_stats(input="significance", flags="cn").text)

global sigma used as fallback: 0.0882 m


Propagated DoD uncertainty: <sigma_dod>
LoD critical value z(0.95) = 1.9600
Color table for raster map <significance> set to 'rules'

-4 20987
-3 11309
-2 12834
-1 54324
0 318490
1 54250
2 13244
3 11787
4 13375


In [22]:
apply_scheme("zscore", "sequential", 0, 10)
panel_figure(
    [
        render("e1", [("d.rast", {"map": "zscore"})]),
        render(
            "e2",
            [
                ("d.rast", {"map": "significance"}),
                (
                    "d.legend",
                    {
                        "raster": "significance",
                        "at": "55,95,3,8",
                        "flags": "cb",
                        "fontsize": 13,
                    },
                ),
            ],
        ),
    ],
    os.path.join(TOOLBOX, "r.dem.errprop", "r_dem_errprop_classes.png"),
    titles=["z-score, capped at 10", "Significance classes, -4 to 4"],
    colorbar=(0, 10, scheme_cmap("sequential")),
    cbar_label="z-score (left panel)",
)

'/home/coreywhite/Documents/GitHub/cwhite911/r-dem/src/raster/r.dem/r.dem.errprop/r_dem_errprop_classes.png'

## 8. Regional screening

Screening runs on a coarser grid: the point is to find where to look, not to
measure what happened.

In [23]:
tools.g_region(raster="elev_lid792_1m", res=10, flags="a")
tools.r_resamp_stats(
    input="dod_significant",
    output="dod_10m_masked",
    method="average",
    overwrite=True,
)
# Blocks holding no significant cell come back NULL. For screening that
# means no change, so make it explicit rather than leaving holes that read
# as missing data.
tools.r_mapcalc(
    expression="dod_10m = if(isnull(dod_10m_masked), 0, dod_10m_masked)", overwrite=True
)
# Topographic triage only. Every input here is real: the DoD comes from the
# surfaces, the road network from the sample dataset.
print(
    tools.r_dem_screen(
        dod="dod_10m",
        output="triage",
        infrastructure="pgcp_roads",
        hazard_output="hazard",
        topo_threshold=1.0,
        overwrite=True,
    ).stderr
)

Triage: topographic change only (no spectral input)
   0%   4%   8%  12%  16%  20%  24%  28%  32%  36%  40%  44%  48%  52%  56%  60%  64%  68%  72%  76%  80%  84%  88%  92%  96% 100%
Triage summary:
Class 0 (No change ): 4,975 cells (0.50 km2)
Class 2 (Topo change ): 275 cells (0.03 km2)
Hazard overlay complete.



In [24]:
# The spectral option needs a bitemporal difference. This dataset has only
# one date of imagery, so the raster below is a stand-in: a real pre-event
# NDVI from the Landsat bands, reduced along the flood corridor and
# perturbed with noise. It shows the option working; it is not evidence that
# fusing spectral evidence finds anything, because the vegetation loss is
# imposed rather than observed.
tools.r_mapcalc(
    expression="ndvi_pre = float(lsat7_2002_40 - lsat7_2002_30)"
    " / float(lsat7_2002_40 + lsat7_2002_30)",
    overwrite=True,
)
tools.r_surf_gauss(
    output="ndvi_noise",
    mean=0,
    sigma=0.05,
    seed=7,
    overwrite=True,
)
tools.r_mapcalc(
    expression="ndvi_change = (ndvi_pre - if(abs(change_truth) > 0.3, 0.30, 0)"
    " + ndvi_noise) - ndvi_pre",
    overwrite=True,
)
print(
    tools.r_dem_screen(
        dod="dod_10m",
        spectral_change="ndvi_change",
        output="triage_fused",
        topo_threshold=1.0,
        spectral_threshold=-0.15,
        overwrite=True,
    ).stderr
)

panel_figure(
    [
        render(
            "t1",
            [
                ("d.rast", {"map": "triage"}),
                (
                    "d.legend",
                    {
                        "raster": "triage",
                        "at": "55,95,3,8",
                        "flags": "cb",
                        "fontsize": 13,
                    },
                ),
            ],
        ),
        render(
            "t2",
            [
                ("d.rast", {"map": "hazard"}),
                (
                    "d.legend",
                    {
                        "raster": "hazard",
                        "at": "55,95,3,8",
                        "flags": "cb",
                        "fontsize": 13,
                    },
                ),
            ],
        ),
    ],
    os.path.join(TOOLBOX, "r.dem.screen", "r_dem_screen_triage.png"),
    titles=[
        "Topographic triage (10 m cells)",
        "Infrastructure hazard overlay (10 m cells)",
    ],
)
tools.g_region(raster="elev_lid792_1m")

Triage: topographic + spectral fusion
   0%   4%   8%  12%  16%  20%  24%  28%  32%  36%  40%  44%  48%  52%  56%  60%  64%  68%  72%  76%  80%  84%  88%  92%  96% 100%
Triage summary:
Class 0 (No change ): 4,965 cells (0.50 km2)
Class 1 (Spectral only ): 10 cells (0.00 km2)
Class 2 (Topo change ): 9 cells (0.00 km2)
Class 3 (Topo+Spectral ): 266 cells (0.03 km2)



ToolResult(returncode=0, stdout='', stderr='')

## 9. Figure inventory

Every manual figure, with its size, so the GRASS style guide's 600 px rule
can be checked at a glance.

In [25]:
for root, _, files in sorted(os.walk(TOOLBOX)):
    for name in sorted(files):
        if name.endswith(".png"):
            path = os.path.join(root, name)
            with Image.open(path) as image:
                size = os.path.getsize(path) // 1024
                print(
                    f"{os.path.relpath(path, TOOLBOX):58s}"
                    f"{image.width:5d} x {image.height:<5d}{size:5d} KB"
                )

r_dem_scene.png                                             600 x 223     97 KB
r_dem_workflow.png                                          600 x 1240   149 KB
r.dem.bias/r_dem_bias_methods.png                           600 x 289    217 KB
r.dem.change/r_dem_change_volumes.png                       600 x 401    135 KB
r.dem.coregister/r_dem_coregister_residuals.png             600 x 285    201 KB
r.dem.errprop/r_dem_errprop_classes.png                     600 x 401    319 KB
r.dem.icp/r_dem_icp_alignment.png                           600 x 401    265 KB
r.dem.lod/r_dem_lod_local.png                               600 x 401    147 KB
r.dem.nk/r_dem_nk_convergence.png                           600 x 401    264 KB
r.dem.screen/r_dem_screen_triage.png                        600 x 333     42 KB
r.dem.stats/r_dem_stats_metrics.png                         600 x 223    178 KB
